In [ ]:
from torch.distributions import MultivariateNormal
import torch
from torch import Tensor


def matrix_normal_sample(M: Tensor, U: Tensor, V: Tensor) -> Tensor:
    """Sample from a matrix normal distribution with mean M and covariance matrices U and V.

    :param M: The mean matrix of shape (n, p).
    :param U: The row covariance matrix of shape (n, n).
    :param V: The column covariance matrix of shape (p, p).
    :return: A sample from the matrix normal distribution of shape (n, p).
    """
    n, p = M.shape
    assert U.shape == (n, n)
    assert V.shape == (p, p)

    L_u = torch.linalg.cholesky(U)
    L_v = torch.linalg.cholesky(V)

    Z = torch.randn(n, p)
    return M + L_u @ Z @ L_v.T

def matrix_normal_kl(
    M_1: Tensor,
    U_1: Tensor,
    V_1: Tensor,
    M_2: Tensor,
    U_2: Tensor,
    V_2: Tensor,
) -> Tensor:
    r"""
    $$
    \mathrm{KL}[P\,||\,Q] &= \frac{1}{2}
        \left[
            \mathrm{vec}(M_2 - M_1)^\mathrm{T} \mathrm{vec}
            \left(U_2^{-1} (M_2 - M_1) V_2^{-1}\right)
        \right. \\
    &+ \left. \mathrm{tr}\left(
        (V_2^{-1}V_1) \otimes (U_2^{-1}U_1)
    \right) - n \ln \frac{|V_1|}{|V_2|} - p \ln \frac{|U_1|}{|U_2|} - n p \right] \; .
    $$

    https://statproofbook.github.io/P/matn-kl.html
    """
    n, p = M_1.shape
    assert M_2.shape == (n, p)
    assert U_1.shape == (n, n)
    assert U_2.shape == (n, n)
    assert V_1.shape == (p, p)
    assert V_2.shape == (p, p)

    M_diff = M_2 - M_1

    # Term 1: vec(M_diff)^T vec(U_2^{-1} M_diff V_2^{-1})
    U2_inv_M_diff = torch.linalg.solve(U_2, M_diff)
    # Right-multiply by V_2^{-1} using transposes: X V_2^{-1} = (V_2^{-T} X^T)^T
    inv_term = torch.linalg.solve(V_2.mT, U2_inv_M_diff.mT).mT
    # vec(A)^T vec(B) is algebraically equivalent to sum(A * B)
    term1 = torch.sum(M_diff * inv_term)

    # Term 2: tr(A \otimes B) = tr(A) * tr(B)
    trace_V = torch.trace(torch.linalg.solve(V_2, V_1))
    trace_U = torch.trace(torch.linalg.solve(U_2, U_1))
    term2 = trace_V * trace_U

    # Terms 3, 4, 5
    term3 = -n * torch.logdet(V_1) + n * torch.logdet(V_2)
    term4 = -p * torch.logdet(U_1) + p * torch.logdet(U_2)
    term5 = -n * p

    return 0.5 * (term1 + term2 + term3 + term4 + term5)